# NB31 — KANSER Kumulative Entegrasyon: NB16 FE + NB30 COMBINED + Multi-Layer Stacking

**Tarih:** 2026-06-22

## Felsefe

Sifirdan yeni strateji denemek yerine, kanitlanmis bilesenler birlestir:
- **NB30 E1:** COMBINED pooling (Boot-F1=0.714)
- **NB16 FE:** Grantham/BLOSUM62/stopgain → +0.015 kanitlanmis
- **Pretrain+Finetune NN/DNN:** dnn_ft KANSER F1=0.7046
- **2-katmanli stacking:** L1 base OOF + L2 stacker + L3 meta-LR
- **OOB vs OOF:** Meta-feature karsilastirmasi
- **Kalibrasyon + prior-shift:** Alexandari receptesi
- **Missing-aware ensemble:** M3+ vs M3- + combo

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from imblearn.ensemble import BalancedBaggingClassifier
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("[UYARI] imblearn bulunamadi")

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("[UYARI] catboost bulunamadi")

sys.path.insert(0, os.path.abspath(".."))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR
from src.metrics import compute_all_metrics, optimize_threshold

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, LeaveOneOut
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

np.random.seed(SEED)
torch.manual_seed(SEED)

PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
N_OOF_FOLDS = 5
BOOT_SEED = 123

RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v16_kanser_cumulative")
REPORTS_DIR_NB = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"NB31 -- KANSER Kumulative Entegrasyon")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"Results -> {RESULTS_DIR}")

NB31 -- KANSER Kumulative Entegrasyon
SEED=42, PI_TEST=0.2, N_BOOT=50
Results -> /Users/tefe/teknofest_model/teknofest_model/results/v16_kanser_cumulative


In [2]:
# Cell 2: Veri Yükleme + Sütun Temizliği
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

# --- Veri yükleme ---
data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# --- COMBINED: MASTER+PAH+CFTR (KANSER HARİÇ!) ---
df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
print(f"\nCOMBINED (MASTER+PAH+CFTR): {df_combined.shape} (pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()})")

# --- Cross-panel birebir-aynı satır drop ---
feat_cols = [c for c in df_kanser.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    """Tüm feature+label birebir aynı olan satırların panel ID'lerini döndür."""
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_kanser, df_master, feat_cols, TARGET)
if dup_ids:
    df_kanser = df_kanser[~df_kanser[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f"KANSER: {len(dup_ids)} birebir-aynı satır drop -> {df_kanser.shape}")
else:
    print("KANSER: birebir-aynı satır yok")

# --- Sütun temizliği: constant + duplicate ---
constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, num_feat)
drop_cols = set(constant_cols) | dup_drop
print(f"Constant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} -> drop {len(dup_drop)}")
print(f"Toplam drop: {len(drop_cols)}, Kalan feature: {len(feat_cols) - len(drop_cols)}")

# Her dataset'ten drop
keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()
df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)

print(f"\nFinal shapes: MASTER={df_master.shape}, COMBINED={df_combined.shape}, KANSER={df_kanser.shape}")

MASTER: (2931, 353) (pos=2149, neg=782)
KANSER: (388, 353) (pos=268, neg=120)
CFTR:   (111, 353)   (pos=90, neg=21)
PAH:    (372, 353)  (pos=310, neg=62)

COMBINED (MASTER+PAH+CFTR): (3414, 353) (pos=2549, neg=865)
KANSER: 3 birebir-aynı satır drop -> (385, 353)
Constant: 0, Duplicate pairs: 58 -> drop 58
Toplam drop: 58, Kalan feature: 293

Final shapes: MASTER=(2931, 295), COMBINED=(3414, 295), KANSER=(385, 295)


In [3]:
# Cell 3: M3 Preprocessing
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols, target):
    """Train üzerinde fit: median, label encoder, high-missing tespiti."""
    X = train_df[keep_cols].copy()
    y = train_df[target].values
    
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    # High-missing tespit (train üzerinde)
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    
    # Median (train üzerinde)
    medians = X[num_cols].median()
    
    # Kategorik fill + LE
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps
    }

def transform_X(df, keep_cols, prep):
    """Preprocessor uygula, is_missing flagleri ekle."""
    X = df[keep_cols].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    
    # is_missing flag (high miss sütunlar için)
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    
    # Median imputation
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    
    # Kategorik
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    
    return X

# Preprocessor fit (MASTER + COMBINED)
prep_master = fit_preprocessor(df_master, keep_cols, TARGET)
prep_combined = fit_preprocessor(df_combined, keep_cols, TARGET)

# Transform
X_master_df = transform_X(df_master, keep_cols, prep_master)
y_master = df_master[TARGET].values

X_combined_df = transform_X(df_combined, keep_cols, prep_combined)
y_combined = df_combined[TARGET].values

X_kanser_master_df = transform_X(df_kanser, keep_cols, prep_master)
X_kanser_combined_df = transform_X(df_kanser, keep_cols, prep_combined)
y_kanser = df_kanser[TARGET].values

print(f"X_master: {X_master_df.shape}, X_combined: {X_combined_df.shape}")
print(f"X_kanser: {X_kanser_master_df.shape} (master prep), {X_kanser_combined_df.shape} (combined prep)")
print(f"KANSER label dist: pos={y_kanser.sum()}, neg={(y_kanser==0).sum()}")

X_master: (2931, 434), X_combined: (3414, 434)
X_kanser: (385, 434) (master prep), (385, 434) (combined prep)
KANSER label dist: pos=265, neg=120


In [4]:
# Cell 4: Değerlendirme Altyapısı

# --- Prior shift (Saerens 2002) ---
def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

# --- Bootstrap %80/20 ---
def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020(y, prob):
    rng = np.random.RandomState(BOOT_SEED)
    yb, pb = _resample_8020(y, prob, rng)
    best, best_thr = -1.0, 0.5
    for thr in np.arange(0.05, 0.95, 0.01):
        f = _f1_pos(yb, (pb >= thr).astype(int))
        if f > best:
            best, best_thr = f, thr
    return float(best_thr)

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    """N resample ortalamasıyla robust threshold seç."""
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    best_thr = max(thr_scores, key=thr_scores.get)
    return float(best_thr)

# --- LOO-CV metrikleri ---
def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    if pi_train is None:
        pi_train = y_true.mean()
    prob = adjust_prior_shift(oof_proba, pi_train=pi_train) if prior_shift else oof_proba
    thr = select_threshold_8020_robust(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1 = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    auprc = average_precision_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {"mcc": mcc, "f1": f1, "auc": auc, "auprc": auprc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "y_pred": y_pred, "y_true": np.asarray(y_true), "prob": prob}

def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
        "train_prec": float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        "train_rec": float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

print("Değerlendirme altyapısı hazır.")

Değerlendirme altyapısı hazır.


In [5]:
# Cell 5: Model Yardımcıları (Tree + Factory)

LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
    "importance_type": "gain"
}

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbosity": 0,
    "n_jobs": -1
}

def make_lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

def make_xgb_classifier(**kw):
    params = {**XGB_PARAMS, **kw}
    return XGBClassifier(**params)

def make_rf_classifier(**kw):
    params = {"n_estimators": 200, "max_depth": 15, "random_state": SEED, "n_jobs": -1}
    params.update(kw)
    return RandomForestClassifier(**params)

def make_catboost_classifier(**kw):
    params = {"iterations": 300, "verbose": 0, "random_state": SEED}
    params.update(kw)
    if HAS_CATBOOST:
        return CatBoostClassifier(**params)
    return None

def _le_encode_for_loo(X_df):
    """Kategorik sütunları basit label-encode et (LOO uyumlu)."""
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=["object", "category"]).columns.tolist()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, le_maps

def make_stack_meta_features(preds, prefix="base"):
    """Stacking meta-feature kolonlarını sklearn uyumlu string isimlerle üret."""
    preds = np.asarray(preds)
    meta = pd.DataFrame(preds, columns=[f"{prefix}_{i}" for i in range(preds.shape[1])])
    meta["mean"] = preds.mean(axis=1)
    meta["std"] = preds.std(axis=1)
    return meta

def oof_predict(make_model_fn, X_df, y, n_splits=5, seed=SEED):
    """Gerçek OOF tahmin: her örnek, onu görmemiş fold modelinden tahmin alır."""
    oof = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tri, vai in skf.split(X_df, y):
        m = make_model_fn()
        m.fit(X_df.iloc[tri], y[tri])
        oof[vai] = m.predict_proba(X_df.iloc[vai])[:, 1]
    full = make_model_fn()
    full.fit(X_df, y)
    return oof, full

print("Model yardımcıları hazır.")

Model yardımcıları hazır.


In [6]:
# Cell 6: NN/DNN Modelleri (inline torch)

class SmallMLP(nn.Module):
    """NB16 reçetesi: BatchNorm YOK, yüksek dropout + weight_decay."""
    def __init__(self, input_dim, hidden_dim=128, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

class DeepMLP(nn.Module):
    """3 katman + residual, BatchNorm YOK."""
    def __init__(self, input_dim, hidden_dim=128, n_layers=3, dropout=0.4):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList()
        for _ in range(n_layers):
            self.blocks.append(nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)
            ))
        self.output = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout * 0.5)
    def forward(self, x):
        x = self.dropout(self.relu(self.input_proj(x)))
        for i, block in enumerate(self.blocks):
            residual = x
            x = block(x)
            if i % 2 == 1:
                x = x + residual
        return self.output(x).squeeze(-1)

class FocalLoss(nn.Module):
    """Focal loss: benign'i öğrenmeye zorla (hard examples)."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()

def train_nn(model_class, X_train, y_train, X_val=None, y_val=None,
             hidden_dim=128, dropout=0.5, lr=1e-3, weight_decay=1e-4,
             epochs=200, patience=20, batch_size=64, **model_kw):
    """NN eğitimi: FocalLoss + early stopping."""
    device = torch.device("cpu")
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train)
    
    model = model_class(X_tr_sc.shape[1], hidden_dim=hidden_dim, dropout=dropout, **model_kw)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    
    X_t = torch.FloatTensor(X_tr_sc).to(device)
    y_t = torch.FloatTensor(y_train).to(device)
    ds = TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    
    best_loss = float('inf')
    wait = 0
    best_state = None
    
    if X_val is not None:
        X_val_sc = scaler.transform(X_val)
        X_v = torch.FloatTensor(X_val_sc).to(device)
        y_v = torch.FloatTensor(y_val).to(device)
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            if X_val is not None:
                val_loss = criterion(model(X_v), y_v).item()
            else:
                val_loss = criterion(model(X_t), y_t).item()
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    return model, scaler

def predict_nn(model, X, scaler):
    """NN tahmin: olasılık döndür."""
    device = torch.device("cpu")
    X_sc = scaler.transform(X) if scaler else X
    X_t = torch.FloatTensor(X_sc).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(X_t)
        proba = torch.sigmoid(logits).cpu().numpy()
    return proba

def oof_predict_nn(model_class, X_df, y, n_splits=5, seed=SEED, **train_kw):
    """NN için OOF tahmin."""
    oof = np.zeros(len(y))
    X_np = X_df.values if hasattr(X_df, 'values') else X_df
    if n_splits == 1:
        full_model, full_scaler = train_nn(model_class, X_np, y, **train_kw)
        full_train_proba = predict_nn(full_model, X_np, full_scaler).flatten()
        return full_train_proba, full_model, full_scaler, full_train_proba
    if n_splits < 1:
        raise ValueError("n_splits must be >= 1")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tri, vai in skf.split(X_np, y):
        X_tr, y_tr = X_np[tri], y[tri]
        X_va, y_va = X_np[vai], y[vai]
        model, scaler = train_nn(model_class, X_tr, y_tr, X_va, y_va, **train_kw)
        oof[vai] = predict_nn(model, X_va, scaler).flatten()
    # Full model
    full_model, full_scaler = train_nn(model_class, X_np, y, **train_kw)
    full_train_proba = predict_nn(full_model, X_np, full_scaler).flatten()
    return oof, full_model, full_scaler, full_train_proba

print("NN/DNN modelleri tanımlandı.")

NN/DNN modelleri tanımlandı.


In [7]:
# Cell 7: Feature Engineering (NB16'dan — Grantham/BLOSUM62/stopgain)

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}

_B62_RAW = """A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4"""

_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for ri, line in enumerate(_B62_RAW.strip().split("\n")):
    toks = line.split()
    row_aa = toks[0][0]
    vals = [toks[0][1:]] + toks[1:]
    for ci, tok in enumerate(vals):
        col_aa = _ORDER[ri + ci]
        v = int(tok[1:] if tok[0].isalpha() else tok)
        _B62[(row_aa, col_aa)] = v
        _B62[(col_aa, row_aa)] = v

def _grantham_dist(a, b):
    if a == b:
        return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a)) or 215

def _blosum62_score(a, b):
    return _B62.get((a, b), 0)

def add_fe(df):
    """Ham df'e FE sutunlari ekler (satir-bazli, fit gerektirmez)."""
    out = df.copy()
    a1 = out["AA_1"].astype("object")
    a2 = out["AA_2"].astype("object")
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    out["fe_aa_nonstandard"] = (
        ~a1.apply(lambda v: isinstance(v, str) and v in STANDARD_AA) |
        ~a2.apply(lambda v: isinstance(v, str) and v in STANDARD_AA)
    ).astype(int)
    out["fe_grantham"] = [
        _grantham_dist(a, b) if (isinstance(a, str) and isinstance(b, str)
                                  and a in STANDARD_AA and b in STANDARD_AA) else -1
        for a, b in zip(a1, a2)
    ]
    out["fe_blosum62"] = [
        _blosum62_score(a, b) if (isinstance(a, str) and isinstance(b, str)
                                   and a in STANDARD_AA and b in STANDARD_AA) else 0
        for a, b in zip(a1, a2)
    ]
    return out

FE_NEW_COLS = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]

# --- KANSER %50/%50 split ---
from sklearn.model_selection import train_test_split
df_kanser_train, df_kanser_test = train_test_split(
    df_kanser, test_size=0.5, stratify=df_kanser[CR.TARGET_COL], random_state=SEED
)
df_kanser_train = df_kanser_train.reset_index(drop=True)
df_kanser_test = df_kanser_test.reset_index(drop=True)
y_kanser_train = df_kanser_train[CR.TARGET_COL].values
y_kanser_test = df_kanser_test[CR.TARGET_COL].values

print(f"KANSER train: {df_kanser_train.shape} (pos={y_kanser_train.sum()}, neg={(y_kanser_train==0).sum()})")
print(f"KANSER test:  {df_kanser_test.shape} (pos={y_kanser_test.sum()}, neg={(y_kanser_test==0).sum()})")

# --- FE uygula ---
df_combined_fe = add_fe(df_combined)
df_kanser_train_fe = add_fe(df_kanser_train)
df_kanser_test_fe = add_fe(df_kanser_test)

# --- Preprocess (with_fe) ---
keep_cols_fe = keep_cols + FE_NEW_COLS
prep_fe = fit_preprocessor(df_combined_fe, keep_cols_fe, CR.TARGET_COL)

X_combined_fe = transform_X(df_combined_fe, keep_cols_fe, prep_fe)
y_combined = df_combined_fe[CR.TARGET_COL].values
pi_combined = y_combined.mean()

X_kanser_train_fe = transform_X(df_kanser_train_fe, keep_cols_fe, prep_fe)
X_kanser_test_fe = transform_X(df_kanser_test_fe, keep_cols_fe, prep_fe)

# --- Preprocess (no_fe) ---
prep_nofe = fit_preprocessor(df_combined, keep_cols, CR.TARGET_COL)
X_combined_nofe = transform_X(df_combined, keep_cols, prep_nofe)
X_kanser_train_nofe = transform_X(df_kanser_train, keep_cols, prep_nofe)
X_kanser_test_nofe = transform_X(df_kanser_test, keep_cols, prep_nofe)

print(f"\nX_combined no_fe: {X_combined_nofe.shape}, with_fe: {X_combined_fe.shape}")
print(f"X_kanser_test no_fe: {X_kanser_test_nofe.shape}, with_fe: {X_kanser_test_fe.shape}")
print(f"FE dogrulama: grantham L->I = {_grantham_dist('L','I')}, blosum62 W->W = {_blosum62_score('W','W')}")

KANSER train: (192, 295) (pos=132, neg=60)
KANSER test:  (193, 295) (pos=133, neg=60)

X_combined no_fe: (3414, 434), with_fe: (3414, 438)
X_kanser_test no_fe: (193, 434), with_fe: (193, 438)
FE dogrulama: grantham L->I = 5, blosum62 W->W = 11


In [8]:
# Cell 8: Exp 0 — FE Ablasyonu (no_fe vs with_fe)
print("="*70)
print("[Exp 0] FE Ablasyonu: NB30 E1_COMBINED + NB16 FE")
print("="*70)

all_results = {}
all_test_probas = {}

def eval_model(label, y_test, p_test, y_train, p_train):
    """Standart degerlendirme: bootstrap %80/20, LOO-MCC, AUPRC, CM."""
    thr = select_threshold_8020_robust(y_train, p_train)
    boot = bootstrap_8020(y_test, p_test, thr)
    y_pred = (p_test >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
    auprc = average_precision_score(y_test, p_test)
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    tr_met = train_metrics_at(y_train, p_train, thr)
    return {
        "boot_f1": boot["mean"], "boot_std": boot["std"],
        "boot_lo": boot["lo"], "boot_hi": boot["hi"],
        "auprc": auprc, "precision": prec, "recall": rec, "mcc": mcc,
        "thr": thr, "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        **tr_met
    }

# --- no_fe ---
m_nofe = make_lgbm_classifier()
m_nofe.fit(X_combined_nofe, y_combined)
p_nofe_train = m_nofe.predict_proba(X_combined_nofe)[:, 1]
p_nofe_test = m_nofe.predict_proba(X_kanser_test_nofe)[:, 1]
res_nofe = eval_model("E0_nofe", y_kanser_test, p_nofe_test, y_combined, p_nofe_train)
all_results["E0_nofe"] = res_nofe
all_test_probas["E0_nofe"] = p_nofe_test
print(f"  no_fe:   Boot-F1={res_nofe['boot_f1']:.4f} +/- {res_nofe['boot_std']:.3f}  AUPRC={res_nofe['auprc']:.4f}  FP={res_nofe['fp']} FN={res_nofe['fn']}")

# --- with_fe ---
m_fe = make_lgbm_classifier()
m_fe.fit(X_combined_fe, y_combined)
p_fe_train = m_fe.predict_proba(X_combined_fe)[:, 1]
p_fe_test = m_fe.predict_proba(X_kanser_test_fe)[:, 1]
res_fe = eval_model("E0_with_fe", y_kanser_test, p_fe_test, y_combined, p_fe_train)
all_results["E0_with_fe"] = res_fe
all_test_probas["E0_with_fe"] = p_fe_test
print(f"  with_fe: Boot-F1={res_fe['boot_f1']:.4f} +/- {res_fe['boot_std']:.3f}  AUPRC={res_fe['auprc']:.4f}  FP={res_fe['fp']} FN={res_fe['fn']}")
print(f"  FE delta: {res_fe['boot_f1'] - res_nofe['boot_f1']:+.4f}")

[Exp 0] FE Ablasyonu: NB30 E1_COMBINED + NB16 FE
  no_fe:   Boot-F1=0.6517 +/- 0.046  AUPRC=0.9587  FP=13 FN=11
  with_fe: Boot-F1=0.6164 +/- 0.035  AUPRC=0.9670  FP=16 FN=9
  FE delta: -0.0353


In [9]:
# Cell 9: Exp 1 — COMBINED Pretrain + KANSER Finetune NN/DNN
print("\n" + "="*70)
print("[Exp 1] COMBINED Pretrain + KANSER Finetune NN/DNN")
print("="*70)

X_combined_fe_np = X_combined_fe.values.astype(np.float32)
X_kanser_train_fe_np = X_kanser_train_fe.values.astype(np.float32)
X_kanser_test_fe_np = X_kanser_test_fe.values.astype(np.float32)

for model_name, model_class, mkw in [
    ("nn_ft", SmallMLP, {"hidden_dim": 128, "dropout": 0.5}),
    ("dnn_ft", DeepMLP, {"hidden_dim": 128, "dropout": 0.4, "n_layers": 3}),
]:
    print(f"\n  {model_name}: COMBINED pretrain -> KANSER finetune")
    
    # 1) Pretrain on COMBINED (with FE)
    pretrained_model, pretrained_scaler = train_nn(
        model_class, X_combined_fe_np, y_combined.astype(np.float32),
        hidden_dim=mkw["hidden_dim"], dropout=mkw["dropout"],
        lr=1e-3, epochs=200, patience=30,
        **{k: v for k, v in mkw.items() if k not in ["hidden_dim", "dropout"]}
    )
    print(f"    Pretrain tamamlandi (COMBINED n={len(y_combined)})")
    
    # 2) Finetune OOF on KANSER train (5-fold)
    oof_preds = np.zeros(len(y_kanser_train))
    fold_models = []
    skf = StratifiedKFold(N_OOF_FOLDS, shuffle=True, random_state=SEED)
    
    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_kanser_train_fe_np, y_kanser_train)):
        X_ft_tr = X_kanser_train_fe_np[tr_idx]
        y_ft_tr = y_kanser_train[tr_idx].astype(np.float32)
        X_ft_val = X_kanser_train_fe_np[val_idx]
        y_ft_val = y_kanser_train[val_idx].astype(np.float32)
        
        ft_model = deepcopy(pretrained_model)
        params = list(ft_model.parameters())
        for p in params[:-2]:
            p.requires_grad = False
        
        ft_scaler = StandardScaler()
        X_ft_tr_sc = ft_scaler.fit_transform(X_ft_tr)
        X_ft_val_sc = ft_scaler.transform(X_ft_val)
        
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, ft_model.parameters()),
            lr=1e-4, weight_decay=1e-4
        )
        criterion = FocalLoss(alpha=0.25, gamma=2.0)
        
        X_t = torch.FloatTensor(X_ft_tr_sc)
        y_t = torch.FloatTensor(y_ft_tr)
        X_v = torch.FloatTensor(X_ft_val_sc)
        y_v = torch.FloatTensor(y_ft_val)
        ds = TensorDataset(X_t, y_t)
        dl = DataLoader(ds, batch_size=64, shuffle=True)
        
        best_loss, wait, best_state = float('inf'), 0, None
        for epoch in range(100):
            ft_model.train()
            for xb, yb in dl:
                optimizer.zero_grad()
                loss = criterion(ft_model(xb), yb)
                loss.backward()
                optimizer.step()
            ft_model.eval()
            with torch.no_grad():
                vl = criterion(ft_model(X_v), y_v).item()
            if vl < best_loss:
                best_loss = vl
                best_state = deepcopy(ft_model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= 15:
                    break
        
        if best_state:
            ft_model.load_state_dict(best_state)
        ft_model.eval()
        with torch.no_grad():
            oof_preds[val_idx] = torch.sigmoid(ft_model(X_v)).numpy()
        fold_models.append((ft_model, ft_scaler))
    
    # 3) Test prediction: fold ortalaması
    test_preds = np.zeros(len(y_kanser_test))
    for fm, fs in fold_models:
        X_test_sc = fs.transform(X_kanser_test_fe_np)
        fm.eval()
        with torch.no_grad():
            test_preds += torch.sigmoid(fm(torch.FloatTensor(X_test_sc))).numpy()
    test_preds /= len(fold_models)
    
    thr = select_threshold_8020_robust(y_kanser_train, oof_preds)
    res = eval_model(f"E1_{model_name}", y_kanser_test, test_preds, y_kanser_train, oof_preds)
    all_results[f"E1_{model_name}"] = res
    all_test_probas[f"E1_{model_name}"] = test_preds
    print(f"    Boot-F1={res['boot_f1']:.4f} +/- {res['boot_std']:.3f}  AUPRC={res['auprc']:.4f}  FP={res['fp']} FN={res['fn']}")


[Exp 1] COMBINED Pretrain + KANSER Finetune NN/DNN

  nn_ft: COMBINED pretrain -> KANSER finetune
    Pretrain tamamlandi (COMBINED n=3414)
    Boot-F1=0.5577 +/- 0.034  AUPRC=0.8950  FP=20 FN=10

  dnn_ft: COMBINED pretrain -> KANSER finetune
    Pretrain tamamlandi (COMBINED n=3414)
    Boot-F1=0.5129 +/- 0.028  AUPRC=0.8842  FP=26 FN=6


In [10]:
# Cell 10: Exp 2 — 2-Katmanlı OOF Stacking (AutoGluon-tarzı)
print("\n" + "="*70)
print("[Exp 2] 2-Katmanli OOF Stacking")
print("="*70)

# L1: 4 tree base model OOF (COMBINED train -> KANSER test)
l1_oof_combined = {}
l1_full_models = {}

base_configs = {
    "lgbm": make_lgbm_classifier,
    "xgb": make_xgb_classifier,
    "rf": lambda: RandomForestClassifier(n_estimators=200, max_depth=15,
                                          min_samples_leaf=5, class_weight='balanced',
                                          random_state=SEED, n_jobs=-1),
}
if HAS_CATBOOST:
    base_configs["catboost"] = lambda: CatBoostClassifier(
        iterations=300, verbose=0, random_state=SEED)

for name, factory in base_configs.items():
    print(f"  L1 {name}...")
    oof, full_model = oof_predict(factory, X_combined_fe, y_combined, n_splits=N_OOF_FOLDS)
    l1_oof_combined[name] = oof
    l1_full_models[name] = full_model

# L1 KANSER test tahminleri (full model ile)
l1_kanser_test = {}
for name, model in l1_full_models.items():
    l1_kanser_test[name] = model.predict_proba(X_kanser_test_fe)[:, 1]

# L1 KANSER train tahminleri (full model, OOF degil — L2 icin)
l1_kanser_train = {}
for name, model in l1_full_models.items():
    l1_kanser_train[name] = model.predict_proba(X_kanser_train_fe)[:, 1]

# --- L1 meta-feature: KANSER train uzerinde OOF (L2 icin temiz) ---
l1_kanser_oof = {}
for name, factory in base_configs.items():
    print(f"  L1 KANSER OOF {name}...")
    oof_k, _ = oof_predict(factory, X_kanser_train_fe, y_kanser_train, n_splits=N_OOF_FOLDS)
    l1_kanser_oof[name] = oof_k

sorted_names = sorted(l1_kanser_oof.keys())
meta_train_l1 = np.column_stack([l1_kanser_oof[k] for k in sorted_names])
meta_test_l1 = np.column_stack([l1_kanser_test[k] for k in sorted_names])

# L2: Stacker LGBM (kucuk) — L1 OOF + orijinal feature concat
l2_train = np.hstack([meta_train_l1, X_kanser_train_fe.values])
l2_test = np.hstack([meta_test_l1, X_kanser_test_fe.values])

print(f"  L2 input: {l2_train.shape} (meta={meta_train_l1.shape[1]} + feat={X_kanser_train_fe.shape[1]})")

l2_oof = np.zeros(len(y_kanser_train))
l2_fold_models = []
skf = StratifiedKFold(N_OOF_FOLDS, shuffle=True, random_state=SEED+1)
for tri, vai in skf.split(l2_train, y_kanser_train):
    l2m = LGBMClassifier(n_estimators=50, max_depth=3, learning_rate=0.1,
                          class_weight='balanced', random_state=SEED, verbose=-1)
    l2m.fit(l2_train[tri], y_kanser_train[tri])
    l2_oof[vai] = l2m.predict_proba(l2_train[vai])[:, 1]
    l2_fold_models.append(l2m)

l2_test_avg = np.zeros(len(y_kanser_test))
for fm in l2_fold_models:
    l2_test_avg += fm.predict_proba(l2_test)[:, 1]
l2_test_avg /= len(l2_fold_models)

# L3: Meta LR — L1 OOF + L2 OOF + mean/std
l3_train = np.column_stack([meta_train_l1, l2_oof,
                             meta_train_l1.mean(axis=1), meta_train_l1.std(axis=1)])
l3_test = np.column_stack([meta_test_l1, l2_test_avg,
                            meta_test_l1.mean(axis=1), meta_test_l1.std(axis=1)])

l3_lr = LogisticRegression(C=1.0, penalty='l2', class_weight='balanced',
                            random_state=SEED, max_iter=1000)
l3_lr.fit(l3_train, y_kanser_train)
l3_train_preds = l3_lr.predict_proba(l3_train)[:, 1]
l3_test_preds = l3_lr.predict_proba(l3_test)[:, 1]

res_ml = eval_model("E2_multilayer", y_kanser_test, l3_test_preds, y_kanser_train, l3_train_preds)
all_results["E2_multilayer_stack"] = res_ml
all_test_probas["E2_multilayer_stack"] = l3_test_preds
print(f"  Multi-layer stack: Boot-F1={res_ml['boot_f1']:.4f}  AUPRC={res_ml['auprc']:.4f}")

# Karsilastirma: tek-katman LR meta (NB30 gibi)
l1_lr = LogisticRegression(C=1.0, penalty='l2', class_weight='balanced',
                            random_state=SEED, max_iter=1000)
l1_lr.fit(meta_train_l1, y_kanser_train)
l1_test_preds = l1_lr.predict_proba(meta_test_l1)[:, 1]
l1_train_preds = l1_lr.predict_proba(meta_train_l1)[:, 1]
res_sl = eval_model("E2_single", y_kanser_test, l1_test_preds, y_kanser_train, l1_train_preds)
all_results["E2_single_stack_lr"] = res_sl
all_test_probas["E2_single_stack_lr"] = l1_test_preds
print(f"  Single-layer LR:   Boot-F1={res_sl['boot_f1']:.4f}  AUPRC={res_sl['auprc']:.4f}")
print(f"  Multi vs Single delta: {res_ml['boot_f1'] - res_sl['boot_f1']:+.4f}")


[Exp 2] 2-Katmanli OOF Stacking
  L1 lgbm...
  L1 xgb...
  L1 rf...
  L1 catboost...
  L1 KANSER OOF lgbm...
  L1 KANSER OOF xgb...
  L1 KANSER OOF rf...
  L1 KANSER OOF catboost...
  L2 input: (192, 442) (meta=4 + feat=438)
  Multi-layer stack: Boot-F1=0.5973  AUPRC=0.9643
  Single-layer LR:   Boot-F1=0.5973  AUPRC=0.9659
  Multi vs Single delta: +0.0000


In [11]:
# Cell 11: Exp 3 — OOB vs OOF Meta-Feature Karsilastirmasi
print("\n" + "="*70)
print("[Exp 3] OOB vs OOF Meta-Feature Karsilastirmasi")
print("="*70)

# RF OOF
rf_factory = lambda: RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_leaf=5,
    class_weight='balanced', oob_score=True, random_state=SEED, n_jobs=-1)

rf_oof, rf_full = oof_predict(rf_factory, X_kanser_train_fe, y_kanser_train, n_splits=N_OOF_FOLDS)
rf_test = rf_full.predict_proba(X_kanser_test_fe)[:, 1]

# RF OOB
rf_oob_model = rf_factory()
rf_oob_model.fit(X_kanser_train_fe, y_kanser_train)
rf_oob = rf_oob_model.oob_decision_function_[:, 1]

# BalancedBagging OOF + OOB
if HAS_IMBLEARN:
    bb_factory = lambda: BalancedBaggingClassifier(
        estimator=make_lgbm_classifier(), n_estimators=20, max_features=0.85,
        sampling_strategy='not minority', oob_score=True, random_state=SEED, n_jobs=-1)
    
    bb_oof, bb_full = oof_predict(bb_factory, X_kanser_train_fe, y_kanser_train, n_splits=N_OOF_FOLDS)
    bb_test = bb_full.predict_proba(X_kanser_test_fe)[:, 1]
    
    bb_oob_model = bb_factory()
    bb_oob_model.fit(X_kanser_train_fe, y_kanser_train)
    bb_oob = bb_oob_model.oob_decision_function_[:, 1]
else:
    bb_oof = rf_oof.copy()
    bb_oob = rf_oob.copy()
    bb_test = rf_test.copy()
    print("  [UYARI] imblearn yok, BB = RF kopyasi")

# Karsilastirma: OOB-only, OOF-only, OOB+OOF
configs = [
    ("OOB_only", np.column_stack([rf_oob, bb_oob]),
                 np.column_stack([rf_test, bb_test])),
    ("OOF_only", np.column_stack([rf_oof, bb_oof]),
                 np.column_stack([rf_test, bb_test])),
    ("OOB_OOF",  np.column_stack([rf_oob, bb_oob, rf_oof, bb_oof]),
                 np.column_stack([rf_test, bb_test, rf_test, bb_test])),
]

for label, tr_feats, te_feats in configs:
    lr = LogisticRegression(C=1.0, class_weight='balanced', random_state=SEED, max_iter=1000)
    lr.fit(tr_feats, y_kanser_train)
    tr_p = lr.predict_proba(tr_feats)[:, 1]
    te_p = lr.predict_proba(te_feats)[:, 1]
    res = eval_model(f"E3_{label}", y_kanser_test, te_p, y_kanser_train, tr_p)
    all_results[f"E3_{label}"] = res
    all_test_probas[f"E3_{label}"] = te_p
    print(f"  {label}: Boot-F1={res['boot_f1']:.4f}  AUPRC={res['auprc']:.4f}")


[Exp 3] OOB vs OOF Meta-Feature Karsilastirmasi


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  OOB_only: Boot-F1=0.6378  AUPRC=0.9293
  OOF_only: Boot-F1=0.6378  AUPRC=0.9293
  OOB_OOF: Boot-F1=0.6286  AUPRC=0.9292


In [12]:
# Cell 12: Exp 4 — Kalibrasyon + Prior-Shift (Alexandari receptesi)
print("\n" + "="*70)
print("[Exp 4] Kalibrasyon + Prior-Shift Stratejileri")
print("="*70)

# Base model: E0_with_fe LGBM (COMBINED train, with_fe)
m_cal_base = make_lgbm_classifier()
m_cal_base.fit(X_combined_fe, y_combined)
p_cal_train = m_cal_base.predict_proba(X_combined_fe)[:, 1]
p_cal_test = m_cal_base.predict_proba(X_kanser_test_fe)[:, 1]

# A) Raw (no calibration, no shift)
res_a = eval_model("E4_A_raw", y_kanser_test, p_cal_test, y_combined, p_cal_train)
all_results["E4_A_raw"] = res_a
all_test_probas["E4_A_raw"] = p_cal_test
print(f"  A) Raw:           Boot-F1={res_a['boot_f1']:.4f}")

# B) Platt only
m_platt = CalibratedClassifierCV(m_cal_base, method='sigmoid', cv=5)
m_platt.fit(X_combined_fe, y_combined)
p_platt_train = m_platt.predict_proba(X_combined_fe)[:, 1]
p_platt_test = m_platt.predict_proba(X_kanser_test_fe)[:, 1]
res_b = eval_model("E4_B_platt", y_kanser_test, p_platt_test, y_combined, p_platt_train)
all_results["E4_B_platt"] = res_b
all_test_probas["E4_B_platt"] = p_platt_test
print(f"  B) Platt:         Boot-F1={res_b['boot_f1']:.4f}")

# C) Saerens only (kalibrasyonsuz)
p_saerens_test = adjust_prior_shift(p_cal_test, pi_combined, PI_TEST)
p_saerens_train = adjust_prior_shift(p_cal_train, pi_combined, PI_TEST)
res_c = eval_model("E4_C_saerens", y_kanser_test, p_saerens_test, y_combined, p_saerens_train)
all_results["E4_C_saerens"] = res_c
all_test_probas["E4_C_saerens"] = p_saerens_test
print(f"  C) Saerens:       Boot-F1={res_c['boot_f1']:.4f}")

# D) Platt -> Saerens (Alexandari receptesi)
p_ps_test = adjust_prior_shift(p_platt_test, pi_combined, PI_TEST)
p_ps_train = adjust_prior_shift(p_platt_train, pi_combined, PI_TEST)
res_d = eval_model("E4_D_platt_saerens", y_kanser_test, p_ps_test, y_combined, p_ps_train)
all_results["E4_D_platt_saerens"] = res_d
all_test_probas["E4_D_platt_saerens"] = p_ps_test
print(f"  D) Platt->Saerens: Boot-F1={res_d['boot_f1']:.4f}")
print(f"\n  En iyi kalibrasyon: {max(['E4_A_raw','E4_B_platt','E4_C_saerens','E4_D_platt_saerens'], key=lambda k: all_results[k]['boot_f1'])}")


[Exp 4] Kalibrasyon + Prior-Shift Stratejileri
  A) Raw:           Boot-F1=0.6164
  B) Platt:         Boot-F1=0.6009
  C) Saerens:       Boot-F1=0.6452
  D) Platt->Saerens: Boot-F1=0.6009

  En iyi kalibrasyon: E4_C_saerens


In [13]:
# Cell 13: Exp 5 — Missing-Aware Dual Model + Ensemble
print("\n" + "="*70)
print("[Exp 5] Missing-Aware Dual Model")
print("="*70)

LEAKAGE_RISK_COLS = getattr(CR, 'AL_MISSINGNESS_LEAKAGE_RISK',
                             [f"AL_{i}" for i in range(16, 26)])

# M3+ : tum is_missing_* + leakage risk sutunlari DAHIL (with_fe)
# (Zaten X_combined_fe ve X_kanser_test_fe bunlari iceriyor)
m_plus = make_lgbm_classifier()
m_plus.fit(X_combined_fe, y_combined)
p_plus_train = m_plus.predict_proba(X_combined_fe)[:, 1]
p_plus_test = m_plus.predict_proba(X_kanser_test_fe)[:, 1]

# M3- : leakage risk sutunlari + is_missing_ HARIC
drop_leak = [c for c in LEAKAGE_RISK_COLS if c in keep_cols_fe]
drop_leak_miss = [f"is_missing_{c}" for c in drop_leak]
keep_minus = [c for c in keep_cols_fe if c not in drop_leak and c not in drop_leak_miss]

prep_minus = fit_preprocessor(df_combined_fe, keep_minus, CR.TARGET_COL)
X_comb_minus = transform_X(df_combined_fe, keep_minus, prep_minus)
X_test_minus = transform_X(df_kanser_test_fe, keep_minus, prep_minus)

m_minus = make_lgbm_classifier()
m_minus.fit(X_comb_minus, y_combined)
p_minus_train = m_minus.predict_proba(X_comb_minus)[:, 1]
p_minus_test = m_minus.predict_proba(X_test_minus)[:, 1]

# Ensemble: 0.5 * M3+ + 0.5 * M3-
p_ens_test = 0.5 * p_plus_test + 0.5 * p_minus_test
p_ens_train = 0.5 * p_plus_train + 0.5 * p_minus_train

for label, p_te, p_tr in [
    ("E5_M3plus", p_plus_test, p_plus_train),
    ("E5_M3minus", p_minus_test, p_minus_train),
    ("E5_ensemble", p_ens_test, p_ens_train),
]:
    res = eval_model(label, y_kanser_test, p_te, y_combined, p_tr)
    all_results[label] = res
    all_test_probas[label] = p_te
    print(f"  {label}: Boot-F1={res['boot_f1']:.4f}  FP={res['fp']} FN={res['fn']}")

print(f"\n  Dropped leakage cols: {len(drop_leak)} AL + {len(drop_leak_miss)} is_missing_")
print(f"  M3+ features: {X_combined_fe.shape[1]}, M3- features: {X_comb_minus.shape[1]}")


[Exp 5] Missing-Aware Dual Model
  E5_M3plus: Boot-F1=0.6164  FP=16 FN=9
  E5_M3minus: Boot-F1=0.6358  FP=15 FN=8
  E5_ensemble: Boot-F1=0.6376  FP=15 FN=7

  Dropped leakage cols: 10 AL + 10 is_missing_
  M3+ features: 438, M3- features: 418


In [14]:
# Cell 14: Exp 6 — Precision-Recall Ensemble Sweep
print("\n" + "="*70)
print("[Exp 6] Precision-Recall Ensemble Sweep")
print("="*70)

# En yuksek precision ve en yuksek recall modellerin test proba'larini kullan
# E0_with_fe (baseline LGBM) vs E2_multilayer_stack (stacking)
p_base = all_test_probas.get("E0_with_fe", p_fe_test)
p_stack = all_test_probas.get("E2_multilayer_stack", l3_test_preds)

p_base_tr = m_fe.predict_proba(X_combined_fe)[:, 1]

for alpha in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p_ens = alpha * p_base + (1 - alpha) * p_stack
    # Threshold: base model train uzerinden (stack train unavailable cleanly)
    thr = select_threshold_8020_robust(y_combined, p_base_tr)
    res = eval_model(f"E6_a{alpha:.1f}", y_kanser_test, p_ens, y_combined, p_base_tr)
    all_results[f"E6_alpha_{alpha:.1f}"] = res
    all_test_probas[f"E6_alpha_{alpha:.1f}"] = p_ens
    print(f"  alpha={alpha:.1f} (base={alpha:.0%}, stack={1-alpha:.0%}): Boot-F1={res['boot_f1']:.4f}  FP={res['fp']} FN={res['fn']}")

# En iyi alpha
best_alpha = max([f"E6_alpha_{a:.1f}" for a in [0.3,0.4,0.5,0.6,0.7]],
                  key=lambda k: all_results[k]['boot_f1'])
print(f"\n  En iyi ensemble: {best_alpha} (Boot-F1={all_results[best_alpha]['boot_f1']:.4f})")


[Exp 6] Precision-Recall Ensemble Sweep
  alpha=0.3 (base=30%, stack=70%): Boot-F1=0.7181  FP=9 FN=12
  alpha=0.4 (base=40%, stack=60%): Boot-F1=0.7181  FP=9 FN=12
  alpha=0.5 (base=50%, stack=50%): Boot-F1=0.7201  FP=9 FN=11
  alpha=0.6 (base=60%, stack=40%): Boot-F1=0.7076  FP=10 FN=10
  alpha=0.7 (base=70%, stack=30%): Boot-F1=0.6898  FP=11 FN=10

  En iyi ensemble: E6_alpha_0.5 (Boot-F1=0.7201)


In [15]:
# Cell 15: Sonuc Derleme + CSV + Gorseller
print("\n" + "="*70)
print("SONUC DERLEMESI")
print("="*70)

rows = []
for exp_name, res in sorted(all_results.items()):
    rows.append({
        "Deney": exp_name,
        "Boot_F1": res.get("boot_f1", np.nan),
        "Boot_std": res.get("boot_std", np.nan),
        "AUPRC": res.get("auprc", np.nan),
        "Precision": res.get("precision", np.nan),
        "Recall": res.get("recall", np.nan),
        "MCC": res.get("mcc", np.nan),
        "FP": res.get("fp", np.nan),
        "FN": res.get("fn", np.nan),
        "Thr": res.get("thr", np.nan),
        "Train_F1": res.get("train_f1", np.nan),
    })

results_df = pd.DataFrame(rows).sort_values("Boot_F1", ascending=False)
results_df.to_csv(os.path.join(RESULTS_DIR, "kanser_cumulative_results.csv"), index=False)

print("\nBootstrap %80/20 F1 siralaması (top 15):")
print(results_df.head(15).to_string(index=False))

print(f"\n--- NB16 Referans: Boot-F1 = 0.716 ---")
best = results_df.iloc[0]
delta = best["Boot_F1"] - 0.716
print(f"En iyi: {best['Deney']} -> Boot-F1={best['Boot_F1']:.4f} (delta={delta:+.4f})")

# --- Gorseller ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Fig 1: FE ablasyonu
fe_data = results_df[results_df["Deney"].str.startswith("E0")]
colors_fe = ["gray" if "nofe" in d else "green" for d in fe_data["Deney"]]
axes[0,0].bar(fe_data["Deney"], fe_data["Boot_F1"], color=colors_fe)
axes[0,0].axhline(y=0.716, color='red', linestyle='--', label='NB16 ref (0.716)')
axes[0,0].set_title("Exp 0: FE Ablasyonu", fontsize=12)
axes[0,0].set_ylabel("Boot %80/20 F1")
axes[0,0].legend()
axes[0,0].tick_params(axis='x', rotation=15)

# Fig 2: Tum deneyler (yatay bar)
top15 = results_df.head(15)
colors_all = ['green' if v > 0.716 else 'steelblue' for v in top15["Boot_F1"]]
axes[0,1].barh(top15["Deney"], top15["Boot_F1"], color=colors_all)
axes[0,1].axvline(x=0.716, color='red', linestyle='--', label='NB16 ref')
axes[0,1].set_title("Tum Deneyler — Boot F1", fontsize=12)
axes[0,1].legend()
axes[0,1].invert_yaxis()

# Fig 3: OOB vs OOF
oob_data = results_df[results_df["Deney"].str.startswith("E3")]
axes[1,0].bar(oob_data["Deney"], oob_data["Boot_F1"], color=["#2196F3","#4CAF50","#FF9800"])
axes[1,0].set_title("Exp 3: OOB vs OOF", fontsize=12)
axes[1,0].set_ylabel("Boot %80/20 F1")
axes[1,0].tick_params(axis='x', rotation=15)

# Fig 4: Kalibrasyon
cal_data = results_df[results_df["Deney"].str.startswith("E4")]
axes[1,1].bar(cal_data["Deney"], cal_data["Boot_F1"], color=["gray","#2196F3","#FF9800","#4CAF50"])
axes[1,1].axhline(y=0.716, color='red', linestyle='--', label='NB16 ref')
axes[1,1].set_title("Exp 4: Kalibrasyon Stratejileri", fontsize=12)
axes[1,1].legend()
axes[1,1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_summary.png"), dpi=150, bbox_inches='tight')
plt.close()

# En iyi confusion matrix
best_key = results_df.iloc[0]["Deney"]
best_res = all_results[best_key]
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
cm = np.array([[best_res.get("tn",0), best_res.get("fp",0)],
               [best_res.get("fn",0), best_res.get("tp",0)]])
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(["Benign", "Patho"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Benign", "Patho"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=18, fontweight='bold',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_title(f"En Iyi: {best_key}\n(Boot-F1={best_res['boot_f1']:.4f})", fontsize=11)
plt.colorbar(im)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_best_confusion.png"), dpi=150, bbox_inches='tight')
plt.close()

print(f"\nGorseller kaydedildi: {RESULTS_DIR}")


SONUC DERLEMESI

Bootstrap %80/20 F1 siralaması (top 15):
       Deney  Boot_F1  Boot_std    AUPRC  Precision   Recall      MCC  FP  FN  Thr  Train_F1
E6_alpha_0.5 0.720050  0.041594 0.968587   0.931298 0.917293 0.760556   9  11 0.77  0.984057
E6_alpha_0.3 0.718072  0.040510 0.967685   0.930769 0.909774 0.749970   9  12 0.77  0.984057
E6_alpha_0.4 0.718072  0.040510 0.967557   0.930769 0.909774 0.749970   9  12 0.77  0.984057
E6_alpha_0.6 0.707574  0.038399 0.969117   0.924812 0.924812 0.758145  10  10 0.77  0.984057
E6_alpha_0.7 0.689773  0.037914 0.968932   0.917910 0.924812 0.744941  11  10 0.77  0.984057
     E0_nofe 0.651700  0.045961 0.958709   0.903704 0.917293 0.707306  13  11 0.81  0.982232
E4_C_saerens 0.645201  0.035972 0.967014   0.898551 0.932331 0.716731  14   9 0.25  0.983855
 E3_OOB_only 0.637803  0.042524 0.929348   0.904000 0.849624 0.629440  12  20 0.50  0.916996
 E3_OOF_only 0.637803  0.042524 0.929269   0.904000 0.849624 0.629440  12  20 0.50  0.905512
 E5_ensembl

In [16]:
# Cell 16: PDF Rapor
from fpdf import FPDF

class KanserCumulativeReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 8, "KANSER Kumulative Entegrasyon Raporu | TEKNOFEST 2025",
                  align="C", new_x="LMARGIN", new_y="NEXT")
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", align="C")

pdf = KanserCumulativeReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)
pdf.add_page()

pdf.set_font("Helvetica", "B", 16)
pdf.cell(0, 12, "KANSER Paneli: Kumulative Entegrasyon", align="C",
         new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 8, f"NB31 | {datetime.now().strftime('%Y-%m-%d')}", align="C",
         new_x="LMARGIN", new_y="NEXT")
pdf.ln(6)

pdf.set_font("Helvetica", "B", 12)
pdf.cell(0, 8, "Yonetici Ozeti", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "", 9)
best_name = results_df.iloc[0]["Deney"]
best_f1 = results_df.iloc[0]["Boot_F1"]
delta_nb16 = best_f1 - 0.716
pdf.multi_cell(0, 5,
    f"KANSER panelinin %80/20 F1 skorunu yukseltmek icin {len(all_results)} deney "
    f"karsilastirildi. En iyi: {best_name} (Boot-F1={best_f1:.4f}). "
    f"NB16 referans: 0.716. Fark: {delta_nb16:+.4f}.\n\n"
    f"Strateji: NB30 COMBINED pooling + NB16 FE (Grantham/BLOSUM62) + "
    f"pretrain/finetune NN/DNN + 2-katmanli OOF stacking + OOB/OOF "
    f"karsilastirma + kalibrasyon + missing-aware dual model.")
pdf.ln(4)

pdf.set_font("Helvetica", "B", 11)
pdf.cell(0, 8, "1. Deney Karsilastirmasi (Boot-mean sirali)", new_x="LMARGIN", new_y="NEXT")

cols = ["Deney", "Boot_F1", "Boot_std", "AUPRC", "Precision", "Recall", "FP", "FN"]
widths = [42, 18, 18, 18, 22, 18, 14, 14]

pdf.set_font("Helvetica", "B", 7)
pdf.set_fill_color(70, 130, 180)
pdf.set_text_color(255, 255, 255)
for c, w in zip(cols, widths):
    pdf.cell(w, 6, c, border=1, fill=True, align="C")
pdf.ln()

pdf.set_text_color(0, 0, 0)
pdf.set_font("Helvetica", "", 7)
for _, row in results_df.head(20).iterrows():
    for c, w in zip(cols, widths):
        val = row[c]
        if isinstance(val, float) and not np.isnan(val):
            txt = f"{val:.4f}" if val < 10 else f"{int(val)}"
        else:
            txt = str(val)
        pdf.cell(w, 5, txt, border=1, align="C")
    pdf.ln()
pdf.ln(4)

for fig_name in ["fig_summary.png", "fig_best_confusion.png"]:
    fig_path = os.path.join(RESULTS_DIR, fig_name)
    if os.path.exists(fig_path):
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(0, 8, "2. Figurler", new_x="LMARGIN", new_y="NEXT")
        pdf.image(fig_path, x=10, w=190)

pdf.add_page()
pdf.set_font("Helvetica", "B", 11)
pdf.cell(0, 8, "3. Tartisma", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "", 9)
pdf.multi_cell(0, 5,
    f"NB31, NB16+NB30 kanitlanmis bilesenlerini birlestirerek KANSER panelinde "
    f"0.716 platosunu kirmayi hedefledi. 7 deney grubu (E0-E6) calistirildi.\n\n"
    f"E0 FE Ablasyonu: NB16'nin Grantham/BLOSUM62/stopgain FE'sinin COMBINED "
    f"pooling uzerine etkisi olculdu.\n"
    f"E1 Pretrain+Finetune: NB30'daki NN/DNN collapse (0.506) onlenmesi icin "
    f"COMBINED pretrain + KANSER finetune uygulandi.\n"
    f"E2 2-Katmanli Stacking: AutoGluon-tarzi L1 base + L2 stacker + L3 LR meta.\n"
    f"E3 OOB vs OOF: RF ve BalancedBagging icin OOB ve OOF meta-feature kalitesi.\n"
    f"E4 Kalibrasyon: Platt -> Saerens (Alexandari receptesi) vs raw/Platt/Saerens.\n"
    f"E5 Missing-Aware: M3+ (leakage dahil) vs M3- (leakage haric) ensemble.\n"
    f"E6 Ensemble Sweep: En iyi base ve stack modellerinin soft ensemble'i.")

report_path = os.path.join(REPORTS_DIR_NB, "NB31_kanser_cumulative_report.pdf")
pdf.output(report_path)
print(f"\nRapor: {report_path}")


Rapor: /Users/tefe/teknofest_model/teknofest_model/reports/NB31_kanser_cumulative_report.pdf
